## Célula 1 — Instalação de dependências

Garante que todas as bibliotecas necessárias estão instaladas no ambiente.

In [49]:
# Execute esta célula apenas uma vez para instalar as dependências
# O '!' permite rodar comandos de terminal direto no Jupyter
#%pip install feedparser google-api-python-client isodate openpyxl python-dotenv --quiet

## Célula 2 — Imports

Carregamos todas as bibliotecas que serão usadas ao longo do notebook.

In [50]:
from __future__ import annotations

import os
import logging
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict, Iterable, List

import feedparser
import isodate
import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build

# ---------------------------------------------------------------------------
# Configuração de log
# Usamos logging ao invés de print() para ter controle de nível (INFO, WARNING, ERROR)
# Isso é especialmente útil em execuções longas: sabemos exatamente o que aconteceu
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

print("✅ Imports OK")

✅ Imports OK


## Célula 3 — Configuração via `.env`

### Por que `.env` e não hardcode?

A API Key **nunca** deve ficar no código-fonte.  
Se você versionar o projeto com Git, a chave ficaria exposta publicamente.  

Crie um arquivo `.env` na mesma pasta do notebook com o conteúdo:
```
YOUTUBE_API_KEY=sua_chave_aqui
```
O `python-dotenv` lê esse arquivo e injeta as variáveis no ambiente.

In [51]:
# Carrega as variáveis do arquivo .env para o ambiente
load_dotenv()

API_KEY: str = os.environ.get("YOUTUBE_API_KEY", "")

# Validação antecipada — melhor falhar cedo do que no meio da execução
if not API_KEY:
    raise EnvironmentError(
        "❌ Variável YOUTUBE_API_KEY não encontrada. "
        "Crie um arquivo .env com: YOUTUBE_API_KEY=sua_chave"
    )

print(f"✅ API Key carregada: {'*' * (len(API_KEY) - 4)}{API_KEY[-4:]}")

✅ API Key carregada: ***********************************6jF4


## Célula 4 — Parâmetros de execução

Centralizamos aqui tudo que o usuário pode querer ajustar.  
Isso evita que parâmetros fiquem espalhados pelo código.

In [52]:
# ──────────────────────────────────────────────
# Parâmetros — edite conforme sua necessidade
# ──────────────────────────────────────────────

CANAIS: List[str] = [
    "UC3Z2XdKUu21_KtMsohZedOQ", "UCN0W0kG9zolEDAn228CvFSQ", "UCCWSZHyZmONoOwb76uMkccg", "UCIRQQEextXQEV18j9kedBOw", "UC-ezuc6BSkMhdPIoaEU8S1g",
    "UCrNt1hZdMx-nFk-tqXsk-GQ", "UCnBNoed6NaPlpL8blj1yy8Q", "UCZE5espnlDl2dDRXD2XLMLg", "UCIt_jZw64ryRDpFW1w-m4vA", "UCcLvYJo2ke6Ckx4CqsEXyRQ",
    "UCkPwI3gaSr65levrx1j9okQ", "UCwLxXLLWEIJFHEeTMlYqHTA", "UCrp0zmecZ3TNV8FSR-tjv7A", "UCD_q9m3EPyFZZ0d3XWcrqWQ", "UCIeL1JF5Q7ALE1qOGPJBm5w",
    "UCj3jEZuLmXOndQJFkPTp3Nw", "UCBX4Z7RAs8ktgsyXm4ES_dg", "UC5Fl2W965fyBCv4gLb-U2hg", "UC_g73GNqsyBWvEehY0lNZBA", "UCD89tNSy9kC6QBjFAcgVd3g",
    "UC-hA65Fjv5X8h-J92zc_L8Q", "UCzLAzI6Q-0WX2IbKfLmtZUw", "UCfMKycqfaMdj9-jVDUuU15Q", "UCn_3RiYrABm3o7puJjTZO3w", "UCGnqRqH9I2v-rNdmXmHmHeg",
    "UChBtPExX9RjCdmpAizK7ccQ", "UC6pkw1C4ha1cmU9aDJ8OxUw", "UCRekFQ2mNTWbMwvH7JdD3xA", "UCQlVe5fDZN99GuRnBKzwn4w", "UCOn6-uNuXeXJqmbywtfBgdQ",
    "UCS4ZSVzsXgDiXV7Pwro4a0A", "UClmrZELCfAbZNzpl6rAT0gw", "UCHGl5Nn0GfMETBgjIkJDWjw", "UCD_q9m3EPyFZZ0d3XWcrqWQ", "UCteJrv6W7MWB5SoRvfjF36Q",
    "UCnTTl5-22qOKRSVmuGwm1Lw", "UCHniugO-y89ZBnaJUCn57Iw", "UCSXdP8V4jRaw-8YMTpa6glw", "UCN0W0kG9zolEDAn228CvFSQ", "UCiuihFCo8gfLInRK55F914g",
    "UCPUb-aMI0HiBjC1rsuRN4ng", "UC3XQ0w18WsYV3vf5E6E7D0w", "UCKjV2NaruwIaBORr9MCcAdw", "UCWTXUWC88wp6Xzr5fTpGEBQ","UCT3iQIbLMQc_wsj87KwlZWA", 
    "UCT0asn_SCNkzi1UDqPRBvUg", "UCUAqQbPNrc2rXx1LRWl5Q2A", "UCCE-jo1GvBJqyj1b287h7jA"
]

MAX_POR_CANAL: int = 10        # quantos vídeos buscar por canal via RSS
LIMITE_SHORTS_SEG: int = 60    # vídeos com duração <= 60s são considerados Shorts
MAX_WORKERS: int = 8           # threads simultâneas para buscar RSS
SAIDA_XLSX: Path = Path("saida/videos_apenas.xlsx")  # caminho de saída

print(f"✅ {len(CANAIS)} canal(is) configurado(s)")
print(f"   Máx. vídeos por canal : {MAX_POR_CANAL}")
print(f"   Limite Shorts         : <= {LIMITE_SHORTS_SEG}s")
print(f"   Workers (RSS paralelo): {MAX_WORKERS}")
print(f"   Saída                 : {SAIDA_XLSX}")

✅ 48 canal(is) configurado(s)
   Máx. vídeos por canal : 10
   Limite Shorts         : <= 60s
   Workers (RSS paralelo): 8
   Saída                 : saida\videos_apenas.xlsx


## Célula 5 — Cliente da YouTube Data API

### Melhoria: instanciar o cliente UMA única vez

Na versão original, `build()` era chamado dentro da função `duracoes_por_video_id()`.  
Isso significa que toda vez que a função fosse chamada, um novo cliente HTTP seria criado.

Aqui criamos o cliente **uma vez** e passamos ele como argumento — mais eficiente e testável.

In [53]:
# Criamos o cliente da API aqui, fora das funções
# Ele será reutilizado em todas as chamadas subsequentes
yt_client = build("youtube", "v3", developerKey=API_KEY)

print("✅ Cliente da YouTube Data API criado com sucesso")

14:45:10  INFO      file_cache is only supported with oauth2client<4.0.0


✅ Cliente da YouTube Data API criado com sucesso


## Célula 6 — Função: buscar um canal via RSS

Separamos a lógica de **um único canal** em uma função própria.  
Isso facilita a paralelização: podemos chamar essa função para N canais ao mesmo tempo.

In [54]:
def _buscar_canal_rss(channel_id: str, max_por_canal: int) -> List[Dict]:
    """
    Busca vídeos de UM único canal via RSS.

    Separamos em função própria para que o ThreadPoolExecutor
    possa chamá-la em paralelo para múltiplos canais.

    Retorna uma lista de dicionários com metadados básicos.
    Em caso de erro no feed, retorna um item com o campo 'error' preenchido.
    """
    url = f"https://www.youtube.com/feeds/videos.xml?channel_id={channel_id}"
    feed = feedparser.parse(url)

    # feedparser sinaliza erros de parsing no atributo 'bozo'
    if getattr(feed, "bozo", 0):
        erro = str(getattr(feed, "bozo_exception", "Erro desconhecido ao ler RSS"))
        log.warning("Erro no feed do canal %s: %s", channel_id, erro)
        return [{
            "channel_id": channel_id,
            "channel_title": None,
            "video_id": None,
            "video_title": None,
            "video_url": None,
            "published": None,
            "error": erro,
        }]

    channel_title = getattr(feed.feed, "title", None)
    itens: List[Dict] = []

    for entry in feed.entries[:max_por_canal]:
        # O RSS do YouTube expõe o video_id no campo 'yt_videoid'
        # Removemos a linha redundante 'video_id = None' que existia no original
        video_id = entry.get("yt_videoid") or entry.get("yt:videoid")

        # Conversão de data: preferimos published_parsed (já parseado pelo feedparser)
        # e só usamos o campo 'published' (string) como fallback
        if getattr(entry, "published_parsed", None):
            dt = datetime(*entry.published_parsed[:6], tzinfo=timezone.utc)
            published_iso = dt.isoformat()
        else:
            published_iso = entry.get("published")

        itens.append({
            "channel_id": channel_id,
            "channel_title": channel_title,
            "video_id": video_id,
            "video_title": entry.get("title"),
            "video_url": entry.get("link"),
            "published": published_iso,
            "error": None,
        })

    log.info("Canal '%s' → %d vídeo(s) encontrado(s)", channel_title or channel_id, len(itens))
    return itens


print("✅ Função _buscar_canal_rss definida")

✅ Função _buscar_canal_rss definida


## Célula 7 — Função: buscar todos os canais em paralelo

### Melhoria: `ThreadPoolExecutor` para paralelização

Na versão original, os canais eram processados **um por um** (loop `for` sequencial).  
Cada `feedparser.parse()` é uma requisição HTTP bloqueante.

Com `ThreadPoolExecutor`, disparamos até `MAX_WORKERS` requisições **ao mesmo tempo**.  
Para 20 canais com latência de ~1s cada: **20s → ~3s** (dependendo da rede).

In [55]:
def buscar_ids_rss(
    channel_ids: Iterable[str],
    max_por_canal: int = 10,
    max_workers: int = 8,
) -> List[Dict]:
    """
    Busca vídeos de TODOS os canais em paralelo usando threads.

    Por que threads e não multiprocessing?
    - A operação é I/O bound (espera de rede), não CPU bound
    - Threads são suficientes e mais leves para esse caso
    - O GIL do Python não é problema aqui

    Retorna lista com todos os vídeos de todos os canais.
    """
    ids = list(channel_ids)
    todos: List[Dict] = []

    log.info("Iniciando busca RSS para %d canal(is) com %d workers...", len(ids), max_workers)

    # submit() envia cada tarefa para o pool de threads
    # as_completed() itera sobre as futures à medida que terminam
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(_buscar_canal_rss, cid, max_por_canal): cid
            for cid in ids
        }

        for future in as_completed(futures):
            channel_id = futures[future]
            try:
                resultado = future.result()
                todos.extend(resultado)
            except Exception as exc:  # noqa: BLE001
                # Captura erros inesperados (timeout, etc.) sem derrubar tudo
                log.error("Falha inesperada no canal %s: %s", channel_id, exc)
                todos.append({
                    "channel_id": channel_id,
                    "channel_title": None,
                    "video_id": None,
                    "video_title": None,
                    "video_url": None,
                    "published": None,
                    "error": str(exc),
                })

    log.info("Total de itens coletados: %d", len(todos))
    return todos


print("✅ Função buscar_ids_rss definida")

✅ Função buscar_ids_rss definida


## Célula 8 — Função: buscar durações via YouTube Data API

### Melhorias aplicadas:
1. **Recebe o cliente** como parâmetro (não cria um novo a cada chamada)
2. **`try/except` por item**: um vídeo com dado faltando não interrompe o lote inteiro
3. **Log de progresso** para lotes grandes

In [56]:
def duracoes_por_video_id(
    cliente,
    video_ids: List[str],
) -> Dict[str, int]:
    """
    Retorna {video_id: duração_em_segundos} para todos os IDs informados.

    A YouTube Data API aceita no máximo 50 IDs por requisição,
    então processamos em lotes de 50.

    Parâmetros
    ----------
    cliente   : objeto retornado por googleapiclient.discovery.build()
    video_ids : lista de IDs de vídeos do YouTube
    """
    out: Dict[str, int] = {}
    total_lotes = (len(video_ids) + 49) // 50  # arredonda para cima

    for i in range(0, len(video_ids), 50):
        lote = video_ids[i : i + 50]
        num_lote = i // 50 + 1
        log.info("Buscando durações — lote %d/%d (%d vídeos)", num_lote, total_lotes, len(lote))

        try:
            resp = (
                cliente.videos()
                .list(part="contentDetails, statistics", id=",".join(lote))
                .execute()
            )
        except Exception as exc:  # noqa: BLE001
            # Erro na chamada HTTP do lote inteiro → loga e pula
            log.error("Erro na API para o lote %d: %s", num_lote, exc)
            continue

        for item in resp.get("items", []):
            vid = item["id"]
            try:
                # A duração vem no formato ISO 8601, ex: 'PT4M13S'
                # isodate.parse_duration converte para timedelta
                iso_dur = item["contentDetails"]["duration"]
                seconds = int(isodate.parse_duration(iso_dur).total_seconds())

                # ── Views ──────────────────────────────────────────────────
                # A API retorna viewCount como string — convertemos para int
                # get() com default '0' evita KeyError se o campo não existir
                # (acontece em alguns vídeos com métricas ocultas)
                view_count = int(item.get("statistics", {}).get("viewCount", 0))

                comment_count = int(item.get('statistics', {}).get('commentCount',0))

                out[vid] = {
                    "duration_seconds": seconds,
                    "view_count": view_count,
                    'comment_count': comment_count,
                }
                
            except (KeyError, AttributeError, ValueError) as exc:
                # Vídeo com dado malformado → registra aviso e continua
                log.warning("Não foi possível parsear duração de '%s': %s", vid, exc)

    log.info("Durações obtidas: %d/%d vídeos", len(out), len(video_ids))
    return out


print("✅ Função duracoes_por_video_id definida")

✅ Função duracoes_por_video_id definida


## Célula 9 — Função: filtrar Shorts

### Melhoria: lógica do NaN corrigida

Na versão original:
```python
df[(df["duration_seconds"].isna()) | (df["duration_seconds"] > limite)]
```
Isso **mantinha** vídeos sem duração (NaN) — potenciais Shorts não filtrados.

Na versão corrigida, descartamos qualquer vídeo sem duração confirmada.

In [57]:
def somente_videos_nao_shorts(
    df: pd.DataFrame,
    cliente,
    limite_shorts_seg: int = 60,
) -> pd.DataFrame:
    """
    Enriquece o DataFrame com a duração de cada vídeo
    e remove Shorts (duração <= limite_shorts_seg).

    Vídeos sem duração retornada pela API são DESCARTADOS
    (comportamento seguro: melhor perder um vídeo do que incluir um Short).
    """
    df = df.copy()

    ids = df["video_id"].dropna().astype(str).unique().tolist()
    if not ids:
        log.warning("Nenhum video_id válido encontrado — retornando DataFrame vazio.")
        return df

    log.info("%d video_id(s) únicos para consultar na API", len(ids))

    dados = duracoes_por_video_id(cliente, ids)
    df["duration_seconds"] = df["video_id"].map(
        {vid: v["duration_seconds"] for vid, v in dados.items()}
    )
    df["view_count"] = df["video_id"].map(
        {vid: v["view_count"] for vid, v in dados.items()}
    )

    df["comment_count"] = df["video_id"].map(
    {vid: v["comment_count"] for vid, v in dados.items()}
    )

    antes = len(df)

    # Corrigido: exigimos que duration_seconds seja > limite
    # NaN (sem duração confirmada) é DESCARTADO
    df = df[df["duration_seconds"] > limite_shorts_seg]

    removidos = antes - len(df)
    log.info("%d Shorts/sem-duração removidos. Restam %d vídeo(s).", removidos, len(df))

    return df


print("✅ Função somente_videos_nao_shorts definida")

✅ Função somente_videos_nao_shorts definida


## Célula 10 — Função: salvar Excel

Função para buscar o numero de inscritos 

In [58]:
def buscar_inscritos(
    cliente,
    channel_ids: List[str],
) -> Dict[str, int]:
    """
    Retorna {channel_id: subscriber_count} para todos os canais informados.

    Usa o endpoint channels().list(part='statistics') da YouTube Data API.
    Assim como vídeos, o limite por requisição é de 50 IDs.

    Canais com contagem oculta (hiddenSubscriberCount=True) retornam 0.
    """
    out: Dict[str, int] = {}

    # Remove duplicatas — vários vídeos podem vir do mesmo canal
    ids_unicos = list(set(channel_ids))
    total_lotes = (len(ids_unicos) + 49) // 50

    for i in range(0, len(ids_unicos), 50):
        lote = ids_unicos[i : i + 50]
        num_lote = i // 50 + 1
        log.info(
            "Buscando inscritos — lote %d/%d (%d canais)",
            num_lote, total_lotes, len(lote),
        )

        try:
            resp = (
                cliente.channels()
                .list(part="statistics", id=",".join(lote))
                .execute()
            )
        except Exception as exc:
            log.error("Erro na API de canais, lote %d: %s", num_lote, exc)
            continue

        for item in resp.get("items", []):
            cid = item["id"]
            stats = item.get("statistics", {})

            # Canais podem ocultar o número de inscritos
            # Nesse caso, hiddenSubscriberCount é True e subscriberCount vem como '0'
            hidden = stats.get("hiddenSubscriberCount", False)
            if hidden:
                log.info("Canal %s oculta inscritos.", cid)

            out[cid] = int(stats.get("subscriberCount", 0))

    log.info("Inscritos obtidos: %d/%d canais", len(out), len(ids_unicos))
    return out


print("✅ Função buscar_inscritos definida")

✅ Função buscar_inscritos definida


## Célula 11 — Função: salvar Excel

Mantida praticamente igual à original — já estava bem implementada.  
Adicionamos apenas log de confirmação.

In [59]:
def salvar_excel(
    df: pd.DataFrame,
    path_xlsx: str | Path,
    sheet_name: str = "videos",
) -> None:
    """
    Salva o DataFrame em arquivo .xlsx.

    O Excel não aceita datas com timezone, então convertemos
    para UTC e depois removemos a informação de fuso.
    """
    path_xlsx = Path(path_xlsx)
    path_xlsx.parent.mkdir(parents=True, exist_ok=True)

    df = df.copy()
    
    # Percorre TODAS as colunas do tipo datetime e remove o timezone
    # Assim funciona independente do nome da coluna (published, Data de publicacao, etc.)
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = (
                pd.to_datetime(df[col], errors="coerce", utc=True)
                .dt.tz_convert(None)
            )

    with pd.ExcelWriter(path_xlsx, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name=sheet_name)

    log.info("✅ Arquivo salvo: %s  (%d linhas)", path_xlsx, len(df))


print("✅ Função salvar_excel definida")

✅ Função salvar_excel definida


## Célula 12 — Função: filtrar vídeos por data e palavras-chave no título

### O que esta função faz

Aplica três filtros em sequência sobre o DataFrame:

**Filtro A — Data (aplicado primeiro):**  
Remove todos os vídeos publicados há mais de `janela_dias` dias.  
Usamos `pd.Timestamp.now(tz='UTC')` para ter a data atual já em UTC,  
o mesmo fuso que o YouTube usa no campo `published`.

**Filtro B — Palavras-chave no título (lógica OU):**  
Mantém apenas vídeos cujo título contenha **pelo menos uma** das palavras-chave.  
Usamos `str.contains(..., case=False)` — isso torna a busca **case-insensitive**,  
ou seja, 'fiis', 'FIIS' e 'FIIs' são tratados da mesma forma.

**Filtro C — Mínimo de views (aplicado por último):**  
Remove vídeos com menos de `min_views` visualizações.  
Vídeos sem `view_count` (NaN) também são descartados — comportamento seguro.

### Bibliotecas utilizadas
- **`pandas`** — `.str.contains()`, comparação de datas e números, `pd.Timestamp`  
- **`datetime`** — `timedelta` para calcular o intervalo de dias  
Nenhuma biblioteca nova — tudo já importado.

In [60]:
def filtrar_por_data_e_titulo(
    df: pd.DataFrame,
    palavras_chave: List[str],
    janela_dias: int = 5,
    min_views: int = 1000,
) -> pd.DataFrame:
    """
    Filtra o DataFrame em duas etapas:

    1. Remove vídeos publicados há mais de `janela_dias` dias.
    2. Mantém apenas vídeos cujo título contenha pelo menos
       uma das `palavras_chave` (busca case-insensitive).
    3. Remove vídeos com menos de `min_views` visualizações.

    Parâmetros
    ----------
    df            : DataFrame com colunas 'published', 'video_title' e 'view_count'
    palavras_chave: lista de termos a buscar no título (ex: ['fiis', 'gare11'])
    janela_dias   : quantos dias para trás considerar a partir de hoje
    min_views     : número mínimo de visualizações (inclusive)
    """
    df = df.copy()

    # ── Etapa 1: Filtro de data ────────────────────────────────────────────
    #
    # Convertemos a coluna 'published' para datetime com UTC
    # (o YouTube sempre publica em UTC — precisamos garantir que estamos
    #  comparando datas no mesmo fuso para não eliminar vídeos errados)
    #
    df["published"] = pd.to_datetime(df["published"], errors="coerce", utc=True)

    # Data de corte: hoje (UTC) menos janela_dias
    from datetime import timedelta
    data_corte = pd.Timestamp.now(tz="UTC") - timedelta(days=janela_dias)

    antes_data = len(df)
    # Mantemos apenas vídeos MAIS RECENTES que a data de corte
    # Vídeos sem data (NaT) são descartados — comportamento seguro
    df = df[df["published"] >= data_corte]
    removidos_data = antes_data - len(df)

    log.info(
        "Filtro de data (últimos %d dias): %d removido(s), %d restante(s)",
        janela_dias, removidos_data, len(df),
    )

    if df.empty:
        log.warning("Nenhum vídeo nos últimos %d dias.", janela_dias)
        return df

    # ── Etapa 2: Filtro de palavras-chave no título (lógica OU) ────────────
    #
    # Para cada palavra-chave, criamos uma máscara booleana (True/False por linha)
    # usando str.contains com case=False (ignora maiúsculas/minúsculas)
    # e na=False (trata título NaN como False, não levanta erro)
    #
    # Em seguida combinamos todas as máscaras com OR (|)
    # Resultado: True se o título contiver QUALQUER uma das palavras
    #
    mascara_titulo = pd.Series(False, index=df.index)  # começa tudo False

    for palavra in palavras_chave:
        mascara_titulo = mascara_titulo | df["video_title"].str.contains(
            palavra, case=False, na=False
        )

    antes_titulo = len(df)
    df = df[mascara_titulo]
    removidos_titulo = antes_titulo - len(df)

    log.info(
        "Filtro de título %s: %d removido(s), %d restante(s)",
        palavras_chave, removidos_titulo, len(df),
    )
    if df.empty:
        log.warning("Nenhum vídeo passou pelo filtro de título.")
        return df

    # ── Etapa 3: Filtro de views mínimas ──────────────────────────────────
    #
    # view_count >= min_views mantém o vídeo
    # NaN (vídeo sem dado de views) é descartado — comportamento seguro
    #
    antes_views = len(df)
    df = df[df["view_count"] >= min_views]
    removidos_views = antes_views - len(df)

    log.info(
        "Filtro de views (>= %d): %d removido(s), %d restante(s)",
        min_views, removidos_views, len(df),
    )

    return df


print("✅ Função filtrar_por_data_e_titulo definida")

✅ Função filtrar_por_data_e_titulo definida


## Célula 13 — Execução principal

Aqui orquestramos tudo na sequência correta, com logs intermediários para acompanhamento.

```
Passo 1 → RSS (paralelo)
Passo 2 → Remove Shorts (API)
Passo 3 → Filtra por data e palavras-chave no título  ← NOVO
Passo 4 → Organiza colunas e exporta para Excel
```

In [61]:
# ── Passo 1: buscar vídeos via RSS (em paralelo) ───────────────────────────
itens = buscar_ids_rss(
    channel_ids=CANAIS,
    max_por_canal=MAX_POR_CANAL,
    max_workers=MAX_WORKERS,
)

df = pd.DataFrame(itens)
print(f"\nDataFrame inicial: {len(df)} linha(s)")
df.head()

14:45:10  INFO      Iniciando busca RSS para 48 canal(is) com 8 workers...
14:45:10  INFO      Canal 'Ramos de Dinheiro' → 10 vídeo(s) encontrado(s)
14:45:10  INFO      Canal 'Academia de Fundos Imobiliários' → 10 vídeo(s) encontrado(s)
14:45:10  INFO      Canal 'A Cara da Riqueza' → 10 vídeo(s) encontrado(s)
14:45:10  INFO      Canal 'Alex Fiis' → 10 vídeo(s) encontrado(s)
14:45:10  INFO      Canal 'ClickInvest' → 10 vídeo(s) encontrado(s)
14:45:10  INFO      Canal 'Renda Com Dividendos | Lucas Santos' → 10 vídeo(s) encontrado(s)
14:45:10  INFO      Canal 'Clube do Valor' → 10 vídeo(s) encontrado(s)
14:45:10  INFO      Canal 'Dani Mengue' → 10 vídeo(s) encontrado(s)
14:45:10  INFO      Canal 'Dia a Dia Investindo' → 10 vídeo(s) encontrado(s)
14:45:10  INFO      Canal 'Dicionário do Investidor' → 10 vídeo(s) encontrado(s)
14:45:10  INFO      Canal 'Dinheiro Com Você - Por William Ribeiro' → 10 vídeo(s) encontrado(s)
14:45:10  INFO      Canal 'EconoMirna' → 10 vídeo(s) encontrado(s)
14:


DataFrame inicial: 470 linha(s)


,channel_id,channel_title,video_id,video_title,video_url,published,error
0,UCN0W0kG9zolEDAn228CvFSQ,Ramos de Dinheiro,BsVJJqcmmLU,Comprei RANI3 e agora tá despencando! COMETI ...,https://www.youtube.com/watch?v=BsVJJqcmmLU,2026-04-30T22:00:13+00:00,None
1,UCN0W0kG9zolEDAn228CvFSQ,Ramos de Dinheiro,rwUkZ-fhF-Y,TGAR11 com 37% de desconto: Oportunidade real ...,https://www.youtube.com/watch?v=rwUkZ-fhF-Y,2026-04-29T22:00:57+00:00,None
2,UCN0W0kG9zolEDAn228CvFSQ,Ramos de Dinheiro,V2mDMinkQVY,"COMPREI NOVA AÇÃO DA CARTEIRA, PAGA 5X DIVID...",https://www.youtube.com/watch?v=V2mDMinkQVY,2026-04-28T22:00:37+00:00,None
3,UCN0W0kG9zolEDAn228CvFSQ,Ramos de Dinheiro,keYlXph35b0,OS 3 MELHORES FIIS BASE 10 PARA INVESTIR? QUAL...,https://www.youtube.com/watch?v=keYlXph35b0,2026-04-27T22:00:51+00:00,None
4,UCN0W0kG9zolEDAn228CvFSQ,Ramos de Dinheiro,ML65PPNbGHg,CMIN3 CAINDO MUITO! 7 Motivos para a Queda da ...,https://www.youtube.com/watch?v=ML65PPNbGHg,2026-04-24T19:10:39+00:00,None


In [62]:
# ── Passo 2: filtrar Shorts via duração (YouTube Data API) ─────────────────
df = somente_videos_nao_shorts(
    df,
    cliente=yt_client,
    limite_shorts_seg=LIMITE_SHORTS_SEG,
)

print(f"\nDataFrame após filtro de Shorts: {len(df)} linha(s)")
df.head()

# ── Passo 2.5: buscar npumero de inscritos por canal (YouTube Data API) ─────────────────
channel_ids_unicos = df["channel_id"].dropna().unique().tolist()
mapa_inscritos = buscar_inscritos(yt_client, channel_ids_unicos)

# Mapeia o subscriber_count para cada linha usando o channel_id
df["subscriber_count"] = df["channel_id"].map(mapa_inscritos)

print(f"\nInscritos mapeados para {len(mapa_inscritos)} canal(is)")

14:45:11  INFO      450 video_id(s) únicos para consultar na API
14:45:11  INFO      Buscando durações — lote 1/9 (50 vídeos)
14:45:12  INFO      Buscando durações — lote 2/9 (50 vídeos)
14:45:12  INFO      Buscando durações — lote 3/9 (50 vídeos)
14:45:12  INFO      Buscando durações — lote 4/9 (50 vídeos)
14:45:12  INFO      Buscando durações — lote 5/9 (50 vídeos)
14:45:13  WARNING   Não foi possível parsear duração de 'ULY5JlSaQio': 'duration'
14:45:13  INFO      Buscando durações — lote 6/9 (50 vídeos)
14:45:13  INFO      Buscando durações — lote 7/9 (50 vídeos)
14:45:13  INFO      Buscando durações — lote 8/9 (50 vídeos)
14:45:13  INFO      Buscando durações — lote 9/9 (50 vídeos)
14:45:14  INFO      Durações obtidas: 449/450 vídeos
14:45:14  INFO      57 Shorts/sem-duração removidos. Restam 413 vídeo(s).
14:45:14  INFO      Buscando inscritos — lote 1/1 (45 canais)
14:45:14  INFO      Inscritos obtidos: 45/45 canais



DataFrame após filtro de Shorts: 413 linha(s)

Inscritos mapeados para 45 canal(is)


In [63]:
# ── Passo 3: filtrar por data e palavras-chave no título ───────────────────
#
# PALAVRAS_CHAVE funciona com lógica OU:
# o vídeo permanece se o título contiver QUALQUER uma das palavras listadas.
# A busca ignora maiúsculas/minúsculas automaticamente.
#
PALAVRAS_CHAVE: List[str] = ["gare11"]
JANELA_DIAS: int = 15  # considera apenas vídeos dos últimos N dias

df = filtrar_por_data_e_titulo(
    df,
    palavras_chave=PALAVRAS_CHAVE,
    janela_dias=JANELA_DIAS,
)

print(f"\nDataFrame após filtro de data e título: {len(df)} linha(s)")
df.head()

14:45:14  INFO      Filtro de data (últimos 15 dias): 155 removido(s), 258 restante(s)
14:45:14  INFO      Filtro de título ['gare11']: 254 removido(s), 4 restante(s)
14:45:14  INFO      Filtro de views (>= 1000): 0 removido(s), 4 restante(s)



DataFrame após filtro de data e título: 4 linha(s)


,channel_id,channel_title,video_id,video_title,video_url,published,error,duration_seconds,view_count,comment_count,subscriber_count
52,UC-ezuc6BSkMhdPIoaEU8S1g,Renda Com Dividendos | Lucas Santos,7h_3n0IvWlE,GGRC11 vs GARE11 | Qual fundo imobiliário rend...,https://www.youtube.com/watch?v=7h_3n0IvWlE,2026-04-30 23:02:09+00:00,None,1384.0,9660.0,117.0,254000
58,UC-ezuc6BSkMhdPIoaEU8S1g,Renda Com Dividendos | Lucas Santos,4yhhNP3Rwsk,GARE11 ou TRXF11? Qual o melhor fundo imobiliá...,https://www.youtube.com/watch?v=4yhhNP3Rwsk,2026-04-25 20:59:57+00:00,None,1494.0,12130.0,97.0,254000
294,UCS4ZSVzsXgDiXV7Pwro4a0A,Liberdade Financeira com Fiis,xNN4L-VJBr8,TRXF11 MXRF11 HGBS11 VGHF11 GARE11 ANÚNCIO DE ...,https://www.youtube.com/watch?v=xNN4L-VJBr8,2026-04-30 22:15:07+00:00,None,588.0,3301.0,15.0,63300
342,UCHniugO-y89ZBnaJUCn57Iw,Pequeno investidor,INjf02PS8oU,GARE11: A Maior OPORTUNIDADE dos FIIs de Tijol...,https://www.youtube.com/watch?v=INjf02PS8oU,2026-04-30 09:00:00+00:00,None,696.0,4983.0,29.0,18100


In [64]:
# ── Passo 4: organizar colunas e exportar ──────────────────────────────────
COLUNAS_FINAIS = [
    "channel_title",
    "subscriber_count",
    "published",
    "duration_seconds",
    "view_count",
    "comment_count",
    "video_title",
    "video_url",
    "video_id",
    "channel_id",
    "error",
]

# Filtramos apenas as colunas que existem no DataFrame
# (evita KeyError se alguma coluna não foi gerada)
df = df[[c for c in COLUNAS_FINAIS if c in df.columns]]

df = df.rename(columns={
    "channel_title"   : "Nome do Canal",
    "subscriber_count": "Inscritos",
    "published"       : "Data de publicacao",
    "duration_seconds": "Duracao",
    "view_count"      : "Views",
    "comment_count"   : "Comentarios",
    "video_title"     : "Titulo do video",
    "video_url"       : "Url do video",
    "video_id"        : "Id video",
    "channel_id"      : "Id Canal",
})

salvar_excel(df, SAIDA_XLSX)

print("\n✅ Pipeline concluído com sucesso!")
df

14:45:14  INFO      ✅ Arquivo salvo: saida\videos_apenas.xlsx  (4 linhas)



✅ Pipeline concluído com sucesso!


,Nome do Canal,Inscritos,Data de publicacao,Duracao,Views,Comentarios,Titulo do video,Url do video,Id video,Id Canal,error
52,Renda Com Dividendos | Lucas Santos,254000,2026-04-30 23:02:09+00:00,1384.0,9660.0,117.0,GGRC11 vs GARE11 | Qual fundo imobiliário rend...,https://www.youtube.com/watch?v=7h_3n0IvWlE,7h_3n0IvWlE,UC-ezuc6BSkMhdPIoaEU8S1g,None
58,Renda Com Dividendos | Lucas Santos,254000,2026-04-25 20:59:57+00:00,1494.0,12130.0,97.0,GARE11 ou TRXF11? Qual o melhor fundo imobiliá...,https://www.youtube.com/watch?v=4yhhNP3Rwsk,4yhhNP3Rwsk,UC-ezuc6BSkMhdPIoaEU8S1g,None
294,Liberdade Financeira com Fiis,63300,2026-04-30 22:15:07+00:00,588.0,3301.0,15.0,TRXF11 MXRF11 HGBS11 VGHF11 GARE11 ANÚNCIO DE ...,https://www.youtube.com/watch?v=xNN4L-VJBr8,xNN4L-VJBr8,UCS4ZSVzsXgDiXV7Pwro4a0A,None
342,Pequeno investidor,18100,2026-04-30 09:00:00+00:00,696.0,4983.0,29.0,GARE11: A Maior OPORTUNIDADE dos FIIs de Tijol...,https://www.youtube.com/watch?v=INjf02PS8oU,INjf02PS8oU,UCHniugO-y89ZBnaJUCn57Iw,None
